<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/TRELLIS_2_4B_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRELLIS.2-4B — A100 image-to-3D

Runs **from GitHub** (`microsoft/TRELLIS.2` + HF weights).  
**Drops off on Google Drive** only: `MyDrive/TRELLIS/outputs/`

Official floor: **24 GB VRAM**. Verified on A100/H100. **T4 will fail — switch Runtime to A100.**

- Model: https://huggingface.co/microsoft/TRELLIS.2-4B  
- Code: https://github.com/microsoft/TRELLIS.2  
- Runtime: Runtime → Change runtime type → **A100 GPU** + High-RAM


In [ ]:
# 0) GPU gate — abort if this is not an A100-class box
import torch, sys
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram_gb:.1f} GB")
if vram_gb < 24:
    raise SystemExit(
        f"NEED >=24GB. This box is {vram_gb:.1f}GB ({name}). "
        "Runtime → Change runtime type → A100. T4 is not enough."
    )
print("A100-class OK.")


In [ ]:
# 1) Drive is OUTPUT ONLY. Code + weights come from GitHub / Hugging Face.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/TRELLIS/outputs")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("Drop-off:", DRIVE_OUT)
print("contents:", list(DRIVE_OUT.iterdir())[:8])


In [ ]:
# 2) Clone official repo from GitHub (source of truth)
import os
os.chdir("/content")
!rm -rf TRELLIS.2
!git clone --recursive -b main https://github.com/microsoft/TRELLIS.2.git
%cd /content/TRELLIS.2
!git log -1 --oneline


In [ ]:
# 3) Install official deps (no conda --new-env — Colab already has Python)
# setup.sh: basic + flash-attn + nvdiffrast + nvdiffrec + cumesh + o-voxel + flexgemm
import os
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
# Colab has torch; skip --new-env so we do not fight the runtime.
!bash -lc 'source ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm'
print("install done")


In [ ]:
# 4) Load pipeline from Hugging Face (not Drive)
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
sys.path.insert(0, "/content/TRELLIS.2")

from trellis2.pipelines import Trellis2ImageTo3DPipeline

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()
print("loaded microsoft/TRELLIS.2-4B  low_vram=", getattr(pipeline, "low_vram", None))


In [ ]:
# 5) Input image — upload, or fall back to the repo example
from google.colab import files
from pathlib import Path
from PIL import Image

INPUT = Path("/content/input.png")
print("Upload a PNG/JPG (cancel / empty = official example T.png)")
uploaded = files.upload()
if uploaded:
    name = next(iter(uploaded))
    Image.open(Path("/content") / name).convert("RGBA").save(INPUT)
    print("using upload:", name)
else:
    src = Path("/content/TRELLIS.2/assets/example_image/T.png")
    Image.open(src).convert("RGBA").save(INPUT)
    print("using example:", src)

img = Image.open(INPUT)
print(img.size, img.mode)
img


In [ ]:
# 6) Generate. A100 default = 512 (fast). Set PIPELINE_TYPE to 1024_cascade if you want the official default.
from pathlib import Path
import time
import cv2
import torch
import imageio
from PIL import Image
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
import o_voxel

PIPELINE_TYPE = "512"          # '512' | '1024' | '1024_cascade' | '1536_cascade'
SEED = 42
DRIVE_OUT = Path("/content/drive/MyDrive/TRELLIS/outputs")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

image = Image.open("/content/input.png")
t0 = time.time()
mesh = pipeline.run(image, seed=SEED, pipeline_type=PIPELINE_TYPE)[0]
print(f"run {time.time()-t0:.1f}s")
mesh.simplify(16777216)

exr = Path("/content/TRELLIS.2/assets/hdri/forest.exr")
envmap = None
if exr.exists():
    envmap = EnvMap(torch.tensor(
        cv2.cvtColor(cv2.imread(str(exr), cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
        dtype=torch.float32, device="cuda",
    ))

stamp = time.strftime("%Y%m%d_%H%M%S")
mp4_path = DRIVE_OUT / f"trellis_{PIPELINE_TYPE}_{stamp}.mp4"
glb_path = DRIVE_OUT / f"trellis_{PIPELINE_TYPE}_{stamp}.glb"

video = render_utils.make_pbr_vis_frames(render_utils.render_video(mesh, envmap=envmap))
imageio.mimsave(str(mp4_path), video, fps=15)

glb = o_voxel.postprocess.to_glb(
    vertices=mesh.vertices,
    faces=mesh.faces,
    attr_volume=mesh.attrs,
    coords=mesh.coords,
    attr_layout=mesh.layout,
    voxel_size=mesh.voxel_size,
    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target=1000000,
    texture_size=2048 if PIPELINE_TYPE == "512" else 4096,
    remesh=True,
    remesh_band=1,
    remesh_project=0,
    verbose=True,
)
glb.export(str(glb_path), extension_webp=True)
print("DROPPED ON DRIVE")
print(mp4_path, mp4_path.stat().st_size)
print(glb_path, glb_path.stat().st_size)
